# Lecture 5 — Exercise 5 (trang 32–34)
**Mục tiêu:** Chọn độ mạnh regularization λ bằng 5-fold cross-validation; đánh giá mô hình cuối cùng **một lần** trên tập test. Gồm (A) dữ liệu Breast Cancer Wisconsin và (B) dữ liệu mô phỏng logistic đa thức.

**Quy ước cho notebook:** tối ưu `-sum(log likelihood) + (λ/2)||β||²` (không phạt intercept). Trong `sklearn.LogisticRegression`, dùng `C=1/λ` khi λ>0, và `C=inf` khi λ=0. **Điểm CV dùng log-loss (càng thấp càng tốt).**

## 1. Chuẩn bị hàm chung
`Pipeline` đặc biệt quan trọng: mỗi lần CV, `StandardScaler` **chỉ học mean/std trên 4 folds dùng để fit**, sau đó áp dụng lên fold validation. Không chuẩn hóa toàn bộ dữ liệu trước CV, vì sẽ rò rỉ thông tin (data leakage).

In [ ]:
import warnings
warnings.filterwarnings('ignore', message=r'Setting penalty=None will ignore', category=UserWarning)

import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, log_loss

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def make_model(lam, degree=None):
    steps = []
    if degree is not None:
        steps.append(('poly', PolynomialFeatures(degree=degree, include_bias=False)))
    steps.extend([
        ('scale', StandardScaler()),
        ('logistic', LogisticRegression(
            C=np.inf if lam == 0 else 1 / lam,
            solver='lbfgs', max_iter=5000
        ))
    ])
    return Pipeline(steps)

def cv_log_loss(model, X, y):
    # scoring='neg_log_loss' trả về số âm: đổi dấu để loss dương
    fold_losses = -cross_val_score(
        model, X, y, cv=cv, scoring='neg_log_loss', n_jobs=1
    )
    return fold_losses.mean(), fold_losses.std()


## 2. Bài A — Breast Cancer Wisconsin (trang 32–33)
1. Nạp 569 quan sát và 30 đặc trưng, y=0 malignant, y=1 benign.
2. Chia 80/20 có `stratify=y`, cố định `random_state=42`.
3. Với 7 λ trong đề, chuẩn hóa bên trong Pipeline rồi 5-fold CV trên **tập train**.
4. Chọn λ có CV log-loss nhỏ nhất, fit lại bằng toàn bộ train, đánh giá test một lần.

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print('Train:', X_train.shape, 'Test:', X_test.shape)

lambda_values = [0, 0.001, 0.01, 0.1, 1, 10, 100]
rows = []
for lam in lambda_values:
    mean_loss, std_loss = cv_log_loss(make_model(lam), X_train, y_train)
    rows.append({'lambda':lam, 'CV log-loss':mean_loss, 'std':std_loss})

cv_real = pd.DataFrame(rows)
print(cv_real.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
best_lam = cv_real.loc[cv_real['CV log-loss'].idxmin(), 'lambda']

final_real = make_model(best_lam).fit(X_train, y_train)
real_accuracy = accuracy_score(y_test, final_real.predict(X_test))
real_log_loss = log_loss(y_test, final_real.predict_proba(X_test))
print(f'\nBest λ = {best_lam:g} | Test accuracy = {real_accuracy:.4f} | Test log-loss = {real_log_loss:.4f}')


Train: (455, 30) Test: (114, 30)
  lambda  CV log-loss    std
  0.0000       1.4624 0.5034
  0.0010       0.6157 0.3971
  0.0100       0.2936 0.2238
  0.1000       0.1218 0.0816
  1.0000       0.0723 0.0289
 10.0000       0.0983 0.0230
100.0000       0.1888 0.0210

Best λ = 1 | Test accuracy = 0.9825 | Test log-loss = 0.0777


### Nhận xét A
- β được fit từ training folds trong mỗi lượt CV; λ được chọn từ validation folds, vẫn nằm bên trong train.
- Sau khi chọn λ, fit lại β trên 455 quan sát train. **Không dùng test để lựa λ hoặc chuẩn hóa**.
- So sánh cột CV log-loss của λ=0 và λ được chọn để bàn về hiệu quả regularization **trong lần chạy này**, không kết luận nó luôn cải thiện mọi bộ dữ liệu.

## 3. Bài B — Dữ liệu mô phỏng (trang 34)
Sinh 600 quan sát từ `X~U[-3,3]` và `Y|X=x~Bernoulli(sigmoid(-1+2x-0.8x²))`. **Bậc thật là d=2** vì log-odds có x và x².

Thử 4 bậc d={1,2,3,5} × 5 λ={0,0.01,0.1,1,10} ⇒ **20 cấu hình**, mỗi cấu hình 5-fold CV. Trong Pipeline: `PolynomialFeatures` → `StandardScaler` → `LogisticRegression`.

In [ ]:
rng = np.random.default_rng(42)
X_sim = rng.uniform(-3, 3, size=(600, 1))
prob = expit(-1 + 2 * X_sim[:,0] - 0.8 * X_sim[:,0]**2)
y_sim = rng.binomial(1, prob)
Xtr, Xte, ytr, yte = train_test_split(
    X_sim, y_sim, test_size=.20, stratify=y_sim, random_state=42
)
print('Simulated train 80%:', Xtr.shape, 'test 20%:', Xte.shape)

degrees = [1, 2, 3, 5]
lambda_sim = [0, 0.01, 0.1, 1, 10]
rows = []
for d in degrees:
    for lam in lambda_sim:
        loss, sd = cv_log_loss(make_model(lam, degree=d), Xtr, ytr)
        rows.append({'degree':d, 'lambda':lam, 'CV log-loss':loss, 'std':sd})
cv_sim = pd.DataFrame(rows)
print('\nCV log-loss (càng thấp càng tốt):')
print(cv_sim.pivot(index='degree', columns='lambda', values='CV log-loss').round(4).to_string())

best_row = cv_sim.loc[cv_sim['CV log-loss'].idxmin()]
best_d, best_lam_sim = int(best_row['degree']), best_row['lambda']
final_sim = make_model(best_lam_sim, degree=best_d).fit(Xtr, ytr)
sim_acc = accuracy_score(yte, final_sim.predict(Xte))
sim_loss = log_loss(yte, final_sim.predict_proba(Xte))
print(f'\nBest (d, λ) = ({best_d}, {best_lam_sim:g}) | Test accuracy = {sim_acc:.4f} | Test log-loss = {sim_loss:.4f}')


Simulated train 80%: (480, 1) test 20%: (120, 1)

CV log-loss (càng thấp càng tốt):
lambda   0.00    0.01    0.10    1.00    10.00
degree                                        
1       0.4740  0.4740  0.4740  0.4740  0.4756
2       0.4187  0.4187  0.4185  0.4186  0.4361
3       0.4217  0.4216  0.4211  0.4200  0.4370
5       0.4259  0.4247  0.4233  0.4220  0.4336

Best (d, λ) = (2, 0.1) | Test accuracy = 0.8083 | Test log-loss = 0.4022


### Nhận xét B và câu hỏi trong slide
Bậc thật của **quá trình sinh dữ liệu** là d=2 nhưng CV đang tối ưu **ước lượng sai số dự đoán từ dữ liệu hữu hạn**, chứ không được tiết lộ công thức sinh dữ liệu. Do nhiễu Bernoulli và việc chia fold, CV **không bắt buộc** chọn d=2 trong mỗi lần chạy.

**Tóm tắt quy trình:** Split → Train-only 5-fold CV (mọi bước biến đổi phải nằm trong folds) → Chọn hyperparameter theo log-loss → Refit toàn bộ train → Test một lần → Nhận xét.